# Gemma4 E2B — KNF/Natural Farming SFT Fine-Tuning

**Runtime:** Google Colab T4 (free tier)  
**Model:** `unsloth/gemma-4-e2b-it` (4-bit QLoRA via Unsloth)  
**Dataset:** `merged_sharegpt.jsonl` hosted on HuggingFace Datasets  
**Output:** LoRA adapter pushed to HF Hub + GGUF (q4_k_m) for CPU inference

---
**Before running:** Set your HuggingFace username in the config cell below.

## 1. Install dependencies

In [ ]:
# Takes ~3-5 min on first run. Unsloth installs its own CUDA-optimised kernels.
!pip install --quiet unsloth trl datasets transformers bitsandbytes peft accelerate huggingface_hub

## 2. Configuration — edit before running

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────
HF_USERNAME        = "YOUR_HF_USERNAME"          # e.g. "johndoe"
HF_DATASET_REPO    = f"{HF_USERNAME}/knf-sharegpt"
HF_ADAPTER_REPO    = f"{HF_USERNAME}/gemma4-natural-farming-lora"
HF_GGUF_REPO       = f"{HF_USERNAME}/gemma4-natural-farming-gguf"
# ─────────────────────────────────────────────────────────────────────────

MODEL_NAME         = "unsloth/gemma-4-e4b-it"
MAX_SEQ_LENGTH     = 1024
LORA_R             = 8
LORA_ALPHA         = 8
LORA_DROPOUT       = 0
BATCH_SIZE         = 1
GRAD_ACCUM         = 8     # effective batch = 8
WARMUP_STEPS       = 100
NUM_EPOCHS         = 3
LEARNING_RATE      = 2e-4
OUTPUT_DIR         = "outputs"
ADAPTER_DIR        = "gemma4-knf-adapter"
GGUF_DIR           = "/tmp/gemma4-knf-gguf"   # /tmp bypasses Kaggle's 20GB working-dir limit

print("Config loaded.")
print(f"  Model        : {MODEL_NAME}")
print(f"  Dataset repo : {HF_DATASET_REPO}")
print(f"  Adapter repo : {HF_ADAPTER_REPO}")
print(f"  GGUF repo    : {HF_GGUF_REPO}")

## 3. HuggingFace login

In [ ]:
from huggingface_hub import login, HfApi
login()  # Enter your HF write token when prompted

## 4. Upload dataset to HuggingFace (run once from local machine or here)

If you haven't pushed `merged_sharegpt.jsonl` yet, run the cell below **once**.
After the upload you can skip this cell on subsequent runs.

In [ ]:
# Upload merged_sharegpt.jsonl — skip if already on HF Hub
import os
from huggingface_hub import HfApi

JSONL_PATH = "merged_sharegpt.jsonl"  # local path in Colab (upload via Files panel first)

if os.path.exists(JSONL_PATH):
    api = HfApi()
    api.create_repo(repo_id=HF_DATASET_REPO, repo_type="dataset", exist_ok=True, private=True)
    api.upload_file(
        path_or_fileobj=JSONL_PATH,
        path_in_repo="train.jsonl",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded to https://huggingface.co/datasets/{HF_DATASET_REPO}")
else:
    print(f"File not found locally: {JSONL_PATH}")
    print("Upload it via the Colab Files panel, then re-run this cell.")

## 5. Load model with 4-bit QLoRA via Unsloth

In [ ]:
import os
import torch
# FastModel is the correct loader for Gemma4 (multimodal).
# FastLanguageModel is for text-only models (Llama, Qwen, etc.) — do NOT use it here.
from unsloth import FastModel

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    dtype=None,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,     # freeze vision encoder — text-only SFT
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,            # 8 per Unsloth's official Gemma4 notebook (16 uses more VRAM)
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

print("Model loaded. Trainable parameters:")
model.print_trainable_parameters()

## 6. Load dataset from HuggingFace

In [ ]:
from datasets import load_dataset

ds = load_dataset(HF_DATASET_REPO, data_files="train.jsonl", split="train")
print(f"Dataset loaded: {len(ds):,} records")
print("Sample record:")
print(ds[0])

## 7. Format dataset with ShareGPT chat template

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="gemma-it")

# Dataset is pre-normalized: strict human/gpt alternation, no system turns.
# standardize_sharegpt maps human→user, gpt→assistant for the tokenizer.
ds = standardize_sharegpt(ds)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    # .removeprefix('<bos>') is required for Gemma4 — the processor tokenizer
    # adds BOS separately; leaving it in the text causes a duplicate.
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

ds = ds.map(formatting_prompts_func, batched=True)
print(f"Formatting done. Training examples: {len(ds):,}")
print(ds[0]["text"][:500])

## 8. Train

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=True,
        bf16=False,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=False,
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining complete. Runtime: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")

## 9. Save LoRA adapter and push to HuggingFace Hub

In [ ]:
# Save locally
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved locally to '{ADAPTER_DIR}'")

# Push to HuggingFace Hub
model.push_to_hub(HF_ADAPTER_REPO)
tokenizer.push_to_hub(HF_ADAPTER_REPO)
print(f"Adapter pushed to https://huggingface.co/{HF_ADAPTER_REPO}")

## 9b. GGUF Recovery — reload adapter from HF Hub (skip if model is still in memory)

Run this block **only** if your Kaggle/Colab session ended after training and the model
is no longer loaded. It reloads the base model and attaches the saved LoRA adapter
from HuggingFace Hub so you can proceed directly to the GGUF export cell.

Skip this entire section if you're continuing in the same session (model already in memory).

In [ ]:
# ── RECOVERY: reload adapter from HF Hub ──────────────────────────────────
# Only run this if your session ended. Skip if model is already in memory.
#
# Pass the adapter repo directly to FastModel.from_pretrained — Unsloth
# detects adapter_config.json and loads base model + LoRA in one shot.
# Do NOT use PeftModel.from_pretrained: it double-wraps the Unsloth model
# and causes key mismatches + a NotImplementedError in save_pretrained_gguf.
# ──────────────────────────────────────────────────────────────────────────
import os, torch
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from huggingface_hub import login

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Fill these in if running standalone (otherwise they come from the config cell above)
HF_USERNAME     = "CopyleftCultivars"
MAX_SEQ_LENGTH  = 1024
HF_ADAPTER_REPO = f"{HF_USERNAME}/gemma4-natural-farming-lora"
HF_GGUF_REPO    = f"{HF_USERNAME}/gemma4-natural-farming-gguf"
GGUF_DIR        = "/tmp/gemma4-knf-gguf"

login()

model, tokenizer = FastModel.from_pretrained(
    model_name=HF_ADAPTER_REPO,   # adapter repo — Unsloth loads base + LoRA automatically
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-it")
print(f"Model + adapter loaded from {HF_ADAPTER_REPO}")

## 10. Export GGUF (q4_k_m) for CPU inference

This merges the LoRA weights into the base model and quantises to GGUF format.  
The resulting file is used by the HuggingFace Spaces chatbot via `llama-cpp-python`.

In [ ]:
# ── DIAGNOSTIC: surface the real llama.cpp stderr before attempting full export ──
# Run this cell FIRST. If you see "unsupported model architecture" or a Python
# import error, the issue is llama.cpp compatibility — restarting won't fix it.
# If you see an out-of-space / OSError, disk is the culprit.
import subprocess, glob, os

converter = "/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py"
test_out  = "/tmp/diag_test.gguf"

# Reuse any temp folder unsloth already extracted to
tmp_dirs = sorted(glob.glob("/tmp/unsloth_gguf_*"), key=os.path.getmtime, reverse=True)
if tmp_dirs:
    src = tmp_dirs[0]
    print(f"Reusing extracted model at: {src}")
else:
    print("No extracted model in /tmp yet — run the export cell once (it will fail),")
    print("then re-run this diagnostic to see the real error.")
    src = None

if src:
    cmd = ["/usr/bin/python3", converter,
           "--outfile", test_out, "--outtype", "f16", src]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("=== STDOUT ===")
    print(result.stdout[-3000:] if result.stdout else "(empty)")
    print("=== STDERR ===")
    print(result.stderr[-3000:] if result.stderr else "(empty)")
    print(f"\nExit code: {result.returncode}")
    if os.path.exists(test_out):
        os.remove(test_out)

# Disk usage
du = subprocess.run(["df", "-h", "/", "/tmp"], capture_output=True, text=True)
print("\n=== DISK USAGE ===")
print(du.stdout)

In [ ]:
# Merge LoRA into base model and push GGUF directly to HF Hub.
# push_to_hub_gguf avoids the local save_pretrained step that hits a
# NotImplementedError in revert_weight_conversion for Gemma4.
#
# temporary_location="/tmp" routes ALL intermediate files (F16 GGUF + quantised
# GGUF) to /tmp, which is separate from Kaggle's 20GB working-dir quota.
# q4_k_m produces ~2.5GB vs ~4GB for q8_0 — fits more comfortably in /tmp.
model.push_to_hub_gguf(
    HF_GGUF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
    temporary_location="/tmp",
)
print(f"GGUF uploaded to https://huggingface.co/{HF_GGUF_REPO}")

## Done!

| Artifact | Location |
|---|---|
| LoRA adapter | `https://huggingface.co/{HF_ADAPTER_REPO}` |
| GGUF model   | `https://huggingface.co/{HF_GGUF_REPO}` |

Next step: deploy `spaces/app.py` to HuggingFace Spaces pointing at the GGUF repo.